In [ ]:
import numpy as np
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import train_test_split

In [ ]:
# loading the row data
df = pd.read_csv("../datasets/Superconductivity/unique_m.csv")
df

In [ ]:
df = df.drop(['critical_temp', 'material'], axis=1)
df

In [ ]:
df = df.sort_index(axis=1)
presence = (df > 0).astype(int)
element_str = presence.apply(lambda row: ''.join(row.index[row == 1]), axis=1)
df['composition'] = element_str
df

In [ ]:
df_train = pd.read_csv("../datasets/Superconductivity/train.csv")
df_train['components'] = df['composition']
df_train

In [ ]:
count_df = df_train.groupby('components').components.count()
df2 = count_df.head(50000).to_frame(name='count')

# df2 = df2[df2['count'] >= 100]
df2 = df2[df2['count']< 100]
df2

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Create the plot
font = {'family': 'monospace', 'size': 16, 'weight':'bold'}
plt.rc('font', **font)
plt.rcParams['font.family'] = 'monospace'

sns.set_style("whitegrid")
plt.figure(figsize=(6, 5), dpi=300)

sns.histplot(data=df2, x='count', kde=True, color='#cdb4db', edgecolor='#e5e5e5',bins=20)

plt.title('Superconductivity-S', fontsize=20, fontname="monospace",weight='bold')
plt.xlabel('Data Group Size', fontsize=18, fontname="monospace", weight='bold')
plt.ylabel('Count', fontsize=18, fontname="monospace", weight='bold')

# Show the plot
plt.tight_layout()
plt.savefig('Super_S_hist.png',
               bbox_inches="tight",
               pad_inches=0.05)
plt.show()

In [ ]:
# pd.set_option('display.max_rows', None)
df2 = df2.sort_values("count")
total = df2['count'].sum()
print(total)
df2

In [ ]:
formulas_list = list(df2.index)
formulas_list

In [ ]:
df_train = df_train.loc[df_train['components'].isin(formulas_list)]

In [ ]:
df_train

In [ ]:
df_subset = df_train.drop(['critical_temp'], axis=1)
df_subset.shape

In [ ]:
seed_split = np.load(file="../seed_for_dataSplit.npy")
seed_inital = np.load(file="../seed_for_initialSamples.npy")
seed_model = np.load(file="../seed_for_model.npy")

In [ ]:
data = df_subset
labels_true = df_train[['critical_temp']]

for i in range(20):
    X_train, X_test, y_train, y_test = train_test_split(data, labels_true, test_size=0.3, random_state=seed_split[i])
    
    scaler = preprocessing.StandardScaler()
    X_train_ = X_train.drop("components", axis='columns')
    X_test_ = X_test.drop("components", axis='columns')
    
    X_train_[X_train_.columns] = scaler.fit_transform(X_train_[X_train_.columns])
    X_test_[X_test_.columns] = scaler.transform(X_test_[X_test_.columns])

    X_train_ = pd.concat((X_train_, X_train[["components"]]), axis=1)
    X_test_ = pd.concat((X_test_, X_test[["components"]]), axis=1)

    X_train_.to_csv("../datasets/Superconductivity/split_data/X_train" + str(i) +".csv", encoding='utf-8', index=False)
    X_test_.to_csv("../datasets/Superconductivity/split_data/X_test" + str(i) +".csv", encoding='utf-8', index=False)
    y_train.to_csv("../datasets/Superconductivity/split_data/y_train" + str(i) +".csv", encoding='utf-8', index=False)
    y_test.to_csv("../datasets/Superconductivity/split_data/y_test" + str(i) +".csv", encoding='utf-8', index=False)